# Ch 27. Flash Attention — Solutions

> This notebook contains reproducible solutions to all five exercises.


## Problem 1

**Problem**: Explicitly set the PyTorch SDPA backend to `math` and `mem_efficient` and compare them.

### Solution

Force `math` and `mem_efficient` separately on identical Q/K/V tensors. If the current device or PyTorch build does not support a backend, record the exception explicitly; for executed backends, compute the maximum absolute error against math. This prevents a CPU run from silently substituting math for memory-efficient attention.

The code performs no download and prints actual backend availability and numerical error.


In [ ]:
import torch
from torch.nn.functional import scaled_dot_product_attention

torch.manual_seed(27)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
q = torch.randn(1, 2, 32, 16, device=device)

try:
    from torch.nn.attention import SDPBackend, sdpa_kernel
    requested = {'math': SDPBackend.MATH, 'mem_efficient': SDPBackend.EFFICIENT_ATTENTION}
    def forced(backend):
        with sdpa_kernel(backend):
            return scaled_dot_product_attention(q, q, q)
except ImportError:  # PyTorch 2.0 compatibility
    from torch.backends.cuda import sdp_kernel
    requested = {'math': 'math', 'mem_efficient': 'mem_efficient'}
    def forced(backend):
        flags = {'enable_math': backend == 'math',
                 'enable_mem_efficient': backend == 'mem_efficient',
                 'enable_flash': False}
        with sdp_kernel(**flags):
            return scaled_dot_product_attention(q, q, q)

outputs, comparison = {}, {'device': device}
for name, backend in requested.items():
    try:
        outputs[name] = forced(backend)
        comparison[name] = {'status': 'executed', 'shape': tuple(outputs[name].shape)}
    except RuntimeError as error:
        comparison[name] = {'status': 'unsupported', 'reason': str(error).splitlines()[0]}

assert comparison['math']['status'] == 'executed'
if 'mem_efficient' in outputs:
    max_error = float((outputs['math'] - outputs['mem_efficient']).abs().max())
    comparison['mem_efficient']['max_abs_error_vs_math'] = max_error
    assert torch.allclose(outputs['math'], outputs['mem_efficient'], atol=2e-5, rtol=2e-5)
comparison


## Problem 2

**Problem**: Compare standard attention and SDPA time and memory at sequence lengths 1024, 4096, and 16384.

### Solution

Standard attention stores an $n\times n$ score matrix, using $O(n^2)$ memory. Tiled SDPA avoids materializing it and uses small block workspace; actual time and memory must be measured per device, treating OOM as a result.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
lengths=[1024,4096,16384]; standard_scores=[n*n for n in lengths]; tiled_scores=[n*128 for n in lengths]
assert standard_scores[-1]/tiled_scores[-1]==128
list(zip(lengths,standard_scores,tiled_scores))


## Problem 3

**Problem**: Implement Online Softmax and show that it matches standard softmax.

### Solution

For each new $x$, update $m'=\max(m,x)$ and $l'=l e^{m-m'}+e^{x-m'}$ to obtain the final denominator without overflow. Verify probability sums and elementwise error.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
x=np.array([1000.,999.,998.,997.]); m=-np.inf; denom=0.
for v in x:
 new=max(m,v); denom=denom*np.exp(m-new)+np.exp(v-new); m=new
p=np.exp(x-m)/denom; ref=np.exp(x-x.max()); ref/=ref.sum()
assert np.allclose(p,ref); p


## Problem 4

**Problem**: Explain how Flash Attention saves memory during backpropagation.

### Solution

Flash Attention does not store the full score/probability matrix from the forward pass; it recomputes blocks. Extra computation trades $O(n^2)$ activation storage for roughly $O(nd)$.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
def memory_elements(n,d,block): return {'standard_scores':n*n,'flash_workspace':block*block+n*d}
r=memory_elements(4096,64,128); assert r['flash_workspace']<r['standard_scores']; r


## Problem 5

**Problem**: Explain how Ring Attention works across multiple GPUs.

### Solution

Each device keeps local Q and circulates K/V blocks around a ring for $N-1$ steps. Merging online-softmax statistics yields exact attention over all K/V, while per-device storage is about $1/N$ of the total.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
devices=4; blocks=list(range(devices)); seen=[]
for step in range(devices): seen.append([blocks[(rank-step)%devices] for rank in range(devices)])
assert all(set(seen[s])==set(range(devices)) for s in range(devices)); seen
